In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import joblib
import base64, io, os, sys
sys.path.append('/app')
from database.db_connection import get_connection

# Load data
conn = get_connection()
df = pd.read_sql("SELECT * FROM ml_features", conn)
conn.close()
df = df[df["price_vnd"].between(500_000, 20_000_000)]
df = df[df["area_m2"].between(6, 80)]

metrics = joblib.load("/app/models/model_metrics_v2.pkl")

DISTRICT_MAP = {
    1:"Quận 1", 2:"Quận 2", 3:"Quận 3", 4:"Quận 4",
    5:"Quận 5", 6:"Quận 6", 7:"Quận 7", 8:"Quận 8",
    9:"Quận 9", 10:"Quận 10", 11:"Quận 11", 12:"Quận 12",
    13:"Bình Thạnh", 14:"Bình Tân", 15:"Gò Vấp",
    16:"Phú Nhuận", 17:"Tân Bình", 18:"Tân Phú",
    19:"Thủ Đức", 20:"Bình Chánh", 21:"Hóc Môn",
    22:"Nhà Bè", 23:"Cần Giờ", 24:"Củ Chi",
}
df["district_name"] = df["district_id"].map(DISTRICT_MAP)

def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=130, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# ── Chart 1: Phân phối giá ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["price_vnd"]/1e6, bins=40, color="#1D9E75", edgecolor="white")
ax.set_xlabel("Gia thue (trieu VND/thang)")
ax.set_ylabel("So luong tin")
ax.set_title("Phan phoi gia thue phong tro TP.HCM")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
chart1 = fig_to_base64(fig)
plt.close()

# ── Chart 2: Giá trung bình theo quận ────────────────────────
district_avg = (
    df.groupby("district_name")["price_vnd"]
    .agg(["mean","count"])
    .sort_values("mean", ascending=True)
)
district_avg["mean_m"] = district_avg["mean"] / 1e6

fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(district_avg.index, district_avg["mean_m"], color="#378ADD")
for i, (idx, row) in enumerate(district_avg.iterrows()):
    ax.text(row["mean_m"]+0.05, i,
            f"n={int(row['count'])}", va="center", fontsize=8)
ax.set_xlabel("Gia trung binh (trieu VND)")
ax.set_title("Gia thue trung binh theo quan - TP.HCM")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
chart2 = fig_to_base64(fig)
plt.close()

# ── Chart 3: Feature importance ───────────────────────────────
features = joblib.load("/app/models/feature_cols_v2.pkl")
model    = joblib.load("/app/models/best_model_v2.pkl")
imp = pd.Series(model.feature_importances_, index=features).nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
imp.plot(kind="barh", ax=ax, color="#7F77DD")
ax.set_title("Top 15 Feature Quan Trong Nhat")
ax.set_xlabel("Importance score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
chart3 = fig_to_base64(fig)
plt.close()

# ── Thống kê tiện ích ─────────────────────────────────────────
amenity_cols = ["has_ac","has_wc","has_furniture","has_washer",
                "has_fridge","has_elevator","has_basement",
                "has_kitchen","has_security","has_parking",
                "no_owner","free_hours"]
amenity_names = ["May lanh","WC rieng","Noi that","May giat",
                 "Tu lanh","Thang may","Ham de xe","Bep",
                 "Bao ve","De xe","Khong chung chu","Gio giac tu do"]
amenity_pct = [df[c].sum()/len(df)*100 if c in df.columns else 0
               for c in amenity_cols]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(amenity_names, amenity_pct, color="#EF9F27")
for i, v in enumerate(amenity_pct):
    ax.text(v+0.5, i, f"{v:.0f}%", va="center", fontsize=9)
ax.set_xlabel("% tin dang co tien ich")
ax.set_title("Ty le tien ich trong cac tin dang")
ax.set_xlim(0, 110)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
chart4 = fig_to_base64(fig)
plt.close()

# ── Tạo HTML report ───────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="vi">
<head>
<meta charset="UTF-8">
<title>SaigonPropTech - Bao Cao Phan Tich Du Lieu</title>
<style>
  body {{ font-family: Arial, sans-serif; max-width: 960px;
          margin: 0 auto; padding: 24px; color: #333; }}
  h1   {{ color: #0F6E56; border-bottom: 2px solid #0F6E56; padding-bottom: 8px; }}
  h2   {{ color: #185FA5; margin-top: 40px; }}
  .metric-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin: 24px 0; }}
  .metric {{ background: #f5f5f5; border-radius: 8px; padding: 16px; text-align: center; }}
  .metric .val {{ font-size: 28px; font-weight: bold; color: #0F6E56; }}
  .metric .lbl {{ font-size: 13px; color: #666; margin-top: 4px; }}
  .model-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; margin: 24px 0; }}
  .model-card {{ background: #e8f5ee; border-radius: 8px; padding: 16px; text-align: center; }}
  .model-card .val {{ font-size: 24px; font-weight: bold; color: #0F6E56; }}
  .model-card .lbl {{ font-size: 13px; color: #555; }}
  img {{ max-width: 100%; border-radius: 8px; margin: 16px 0; }}
  table {{ width: 100%; border-collapse: collapse; margin: 16px 0; }}
  th {{ background: #0F6E56; color: white; padding: 10px; text-align: left; }}
  td {{ padding: 8px 10px; border-bottom: 1px solid #eee; }}
  tr:nth-child(even) {{ background: #f9f9f9; }}
  .footer {{ margin-top: 48px; text-align: center; color: #999; font-size: 12px; }}
</style>
</head>
<body>

<h1>SaigonPropTech — Bao Cao Phan Tich Du Lieu Phong Tro TP.HCM</h1>
<p>Du lieu thu thap tu phongtro123.com · Phan tich ngay 08/04/2026</p>

<h2>1. Tong Quan Du Lieu</h2>
<div class="metric-grid">
  <div class="metric">
    <div class="val">{len(df):,}</div>
    <div class="lbl">Tin dang hop le</div>
  </div>
  <div class="metric">
    <div class="val">{df['district_name'].nunique()}</div>
    <div class="lbl">Quan/Huyen</div>
  </div>
  <div class="metric">
    <div class="val">{df['price_vnd'].median()/1e6:.1f}M</div>
    <div class="lbl">Gia trung vi (VND)</div>
  </div>
  <div class="metric">
    <div class="val">{df['area_m2'].median():.0f}m²</div>
    <div class="lbl">Dien tich trung vi</div>
  </div>
</div>

<h2>2. Phan Phoi Gia Thue</h2>
<img src="data:image/png;base64,{chart1}" alt="Phan phoi gia">

<table>
  <tr><th>Chi so</th><th>Gia tri</th></tr>
  <tr><td>Gia thap nhat</td><td>{df['price_vnd'].min()/1e6:.1f} trieu/thang</td></tr>
  <tr><td>Gia cao nhat</td><td>{df['price_vnd'].max()/1e6:.1f} trieu/thang</td></tr>
  <tr><td>Gia trung binh</td><td>{df['price_vnd'].mean()/1e6:.2f} trieu/thang</td></tr>
  <tr><td>Gia trung vi</td><td>{df['price_vnd'].median()/1e6:.1f} trieu/thang</td></tr>
  <tr><td>Do lech chuan</td><td>{df['price_vnd'].std()/1e6:.2f} trieu</td></tr>
</table>

<h2>3. Gia Thue Theo Quan</h2>
<img src="data:image/png;base64,{chart2}" alt="Gia theo quan">

<h2>4. Ty Le Tien Ich</h2>
<img src="data:image/png;base64,{chart4}" alt="Tien ich">

<h2>5. Mo Hinh Du Doan</h2>
<div class="model-grid">
  <div class="model-card">
    <div class="val">{metrics['r2']*100:.1f}%</div>
    <div class="lbl">R² Score<br>(giai thich bien dong gia)</div>
  </div>
  <div class="model-card">
    <div class="val">{metrics['mae']/1e6:.2f}M</div>
    <div class="lbl">MAE<br>(sai so trung binh)</div>
  </div>
  <div class="model-card">
    <div class="val">{metrics['rmse']/1e6:.2f}M</div>
    <div class="lbl">RMSE<br>(sai so can bac hai)</div>
  </div>
</div>

<table>
  <tr><th>Thong so</th><th>Gia tri</th></tr>
  <tr><td>Thuat toan</td><td>Gradient Boosting Regressor</td></tr>
  <tr><td>So mau train</td><td>{metrics['n_train']:,}</td></tr>
  <tr><td>So mau test</td><td>{metrics['n_test']:,}</td></tr>
  <tr><td>So feature su dung</td><td>{metrics['n_features']}</td></tr>
</table>

<h2>6. Feature Quan Trong Nhat</h2>
<img src="data:image/png;base64,{chart3}" alt="Feature importance">

<div class="footer">
  SaigonPropTech ML Project · Du lieu: phongtro123.com · Mo hinh: Gradient Boosting
</div>
</body>
</html>"""

os.makedirs("/app/data/reports", exist_ok=True)
with open("/app/data/reports/data_report.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Bao cao da luu tai: /app/data/reports/data_report.html")
print("Mo file trong trinh duyet de xem!")

ModuleNotFoundError: No module named 'database'